In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from pathlib import Path
from PIL import Image
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report


In [5]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [6]:
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 0.001
NUM_CLASSES = 7
IMG_SIZE = 224
TRAIN_DIR = Path('sorted/train')
VALID_DIR = Path('sorted/valid')

In [7]:
# dataloaders

class HairLossDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        "Dataset Init"
        self.image_dir = img_dir
        self.transform = transform
        self.classes = [f'class_{i}' for i in range(1, NUM_CLASSES+1)]

        self.images = []
        self.labels = []

        for class_idx, class_name in enumerate(self.classes):
            class_path = os.path.join(self.image_dir, class_name)
            if not os.path.exists(class_path):
                raise ValueError(f"Directory {class_path} doesn't exist")
            
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    self.images.append(os.path.join(class_path, img_name))
                    self.labels.append(class_idx)
        
        print(f'Loaded {len(self.images)} pictures from {len(self.classes)} from path {img_dir}')

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx) -> Any:
        'Loading and transforming image by index'
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        
        return image, label

In [8]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [9]:
def create_dataloaders(train_dir, val_dir, batch_size=32):
    train_dataset = HairLossDataset(
        img_dir=train_dir,
        transform=data_transforms['train']
    )

    val_dataset = HairLossDataset(
        img_dir=val_dir,
        transform=data_transforms['val']
    )

    # making dataloaders
    dataloaders = {
        'train': DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=4
        ),
        'val': DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=4
        )
    }

    dataset_sizes = {
        'train': len(train_dataset),
        'val': len(val_dataset)
    }

    return dataloaders, dataset_sizes

In [1]:
def train_model(
        model,
        loss_function,
        optimizer,
        scheduler,
        dataloaders,
        dataset_sizes,
        epochs=25):

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [] 
    }
    best_model_wts = model.state_dict()
    best_acc = 0.0

    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for stage in ['train', 'val']:
            if stage == 'train':
                model.train()
            else:
                model.eval()
            
            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[stage]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(stage == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = loss_function(outputs, labels)

                    # Backward pass + optimize only if in training stage
                    if stage == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if stage == 'train' and scheduler:
                scheduler.step()
            
            epoch_loss = running_loss / dataset_sizes[stage]
            epoch_acc = running_corrects.double() / dataset_sizes[stage]

            if stage == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
            
            print(f'{stage} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if stage == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                print('Best model weights updated\n')

    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)

    return model, history

In [3]:
def test_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, all_labels

In [10]:
# Loading data
dataloaders, dataset_sizes = create_dataloaders(TRAIN_DIR, VALID_DIR, BATCH_SIZE)

Loaded 5029 pictures from 7 from path sorted\train
Loaded 506 pictures from 7 from path sorted\valid


In [16]:
# Load the model
weights = models.MobileNet_V2_Weights.IMAGENET1K_V2
model = models.mobilenet_v2(weights=weights, progress=True)
num_classes = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_classes, NUM_CLASSES)
model = model.to(device)
print('Model MobileNetV2 loaded')
# print('Model summary:\n', model)

Model MobileNetV2 loaded


In [1]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

NameError: name 'nn' is not defined

In [ ]:
print('Starting training...')
model, history = train_model(
    model,
    loss_function,
    optimizer,
    scheduler,
    dataloaders,
    dataset_sizes,
    NUM_EPOCHS
)
print('Training finished!')
# Save the model
torch.save(model.state_dict(), 'model/model_weights.pth')

Starting training...
Epoch 1/30
----------
